In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
import shutil

PROJECT_DIR = Path(
    "/content/drive/MyDrive/IJEEE_VC_data"
)

PROJECT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

joe_files = list(
    Path("/content").glob("*joe*.gz")
)

kathleen_files = list(
    Path("/content").glob("*kathleen*.gz")
)

assert joe_files, "❌ ملف Joe غير موجود داخل /content"
assert kathleen_files, "❌ ملف Kathleen غير موجود داخل /content"

joe_uploaded = joe_files[0]
kathleen_uploaded = kathleen_files[0]

JOE_ARCHIVE = PROJECT_DIR / "joe.tar.gz"
KATHLEEN_ARCHIVE = PROJECT_DIR / "kathleen.tar.gz"

shutil.copy2(
    joe_uploaded,
    JOE_ARCHIVE
)

shutil.copy2(
    kathleen_uploaded,
    KATHLEEN_ARCHIVE
)

print("✅ Joe saved:", JOE_ARCHIVE)
print("✅ Kathleen saved:", KATHLEEN_ARCHIVE)
print("Joe size:", round(JOE_ARCHIVE.stat().st_size / 1024**2, 2), "MB")
print("Kathleen size:", round(KATHLEEN_ARCHIVE.stat().st_size / 1024**2, 2), "MB")

✅ Joe saved: /content/drive/MyDrive/IJEEE_VC_data/joe.tar.gz
✅ Kathleen saved: /content/drive/MyDrive/IJEEE_VC_data/kathleen.tar.gz
Joe size: 75.78 MB
Kathleen size: 211.96 MB


In [ ]:
import subprocess
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/IJEEE_VC_data"
)

JOE_ARCHIVE = PROJECT_DIR / "joe.tar.gz"
KATHLEEN_ARCHIVE = PROJECT_DIR / "kathleen.tar.gz"

DATASET_DIR = Path("/content/hybrid_dataset")
JOE_DIR = DATASET_DIR / "joe"
KATHLEEN_DIR = DATASET_DIR / "kathleen"

JOE_DIR.mkdir(parents=True, exist_ok=True)
KATHLEEN_DIR.mkdir(parents=True, exist_ok=True)

assert JOE_ARCHIVE.exists(), "❌ Joe archive not found"
assert KATHLEEN_ARCHIVE.exists(), "❌ Kathleen archive not found"

print("Extracting Joe dataset...")

subprocess.run(
    [
        "tar",
        "--no-same-owner",
        "-xzf",
        str(JOE_ARCHIVE),
        "-C",
        str(JOE_DIR)
    ],
    check=True
)

print("Extracting Kathleen dataset...")

subprocess.run(
    [
        "tar",
        "--no-same-owner",
        "-xzf",
        str(KATHLEEN_ARCHIVE),
        "-C",
        str(KATHLEEN_DIR)
    ],
    check=True
)

joe_audio = list(JOE_DIR.rglob("*.webm"))
joe_text = list(JOE_DIR.rglob("*.txt"))

kathleen_audio = list(
    KATHLEEN_DIR.rglob("*.flac")
)

kathleen_text = list(
    KATHLEEN_DIR.rglob("*.txt")
)

print("\n✅ Extraction completed")
print("Joe audio files:", len(joe_audio))
print("Joe text files:", len(joe_text))
print(
    "Kathleen audio files:",
    len(kathleen_audio)
)
print(
    "Kathleen text files:",
    len(kathleen_text)
)

Extracting Joe dataset...
Extracting Kathleen dataset...

✅ Extraction completed
Joe audio files: 1150
Joe text files: 1150
Kathleen audio files: 1620
Kathleen text files: 1620


In [ ]:
import subprocess
import shutil
import pandas as pd
import soundfile as sf
from pathlib import Path

JOE_DIR = Path("/content/hybrid_dataset/joe")
KATHLEEN_DIR = Path(
    "/content/hybrid_dataset/kathleen"
)

SELECTED_DIR = Path(
    "/content/selected_hybrid_data"
)

DRIVE_SELECTED_DIR = Path(
    "/content/drive/MyDrive/"
    "IJEEE_VC_data/selected_hybrid_data"
)

SELECTED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DRIVE_SELECTED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Same fixed samples used previously
selected_files = [
    {
        "role": "source",
        "sample": 1,
        "input": (
            JOE_DIR /
            "en-US/0000000001_0300000050_General/"
            "0000000160.webm"
        ),
        "output": (
            SELECTED_DIR /
            "joe_source_01.wav"
        )
    },
    {
        "role": "source",
        "sample": 2,
        "input": (
            JOE_DIR /
            "en-US/0000000001_0300000050_General/"
            "0000000229.webm"
        ),
        "output": (
            SELECTED_DIR /
            "joe_source_02.wav"
        )
    },
    {
        "role": "source",
        "sample": 3,
        "input": (
            JOE_DIR /
            "en-US/0000000001_0300000050_General/"
            "0000000068.webm"
        ),
        "output": (
            SELECTED_DIR /
            "joe_source_03.wav"
        )
    },
    {
        "role": "target_reference",
        "sample": 0,
        "input": (
            KATHLEEN_DIR /
            "en_US/arctic_a0407_1592707592.flac"
        ),
        "output": (
            SELECTED_DIR /
            "kathleen_reference.wav"
        )
    }
]


def convert_to_wav(input_path, output_path):
    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-loglevel", "error",
            "-i", str(input_path),
            "-ac", "1",
            "-ar", "22050",
            "-c:a", "pcm_s16le",
            str(output_path)
        ],
        check=True
    )


manifest_records = []

for item in selected_files:
    input_path = item["input"]
    output_path = item["output"]

    assert input_path.exists(), (
        f"❌ Missing input: {input_path}"
    )

    convert_to_wav(
        input_path,
        output_path
    )

    transcript_path = input_path.with_suffix(
        ".txt"
    )

    transcript = (
        transcript_path.read_text(
            encoding="utf-8",
            errors="ignore"
        ).strip()
        if transcript_path.exists()
        else ""
    )

    duration = sf.info(
        str(output_path)
    ).duration

    manifest_records.append({
        "role": item["role"],
        "sample": item["sample"],
        "original_file": str(input_path),
        "prepared_wav": str(output_path),
        "duration_seconds": duration,
        "transcript": transcript
    })


manifest_df = pd.DataFrame(
    manifest_records
)

manifest_path = (
    SELECTED_DIR /
    "selected_samples_manifest.csv"
)

manifest_df.to_csv(
    manifest_path,
    index=False
)

# Save the fixed inputs to Drive
shutil.copytree(
    SELECTED_DIR,
    DRIVE_SELECTED_DIR,
    dirs_exist_ok=True
)

print("✅ Fixed samples prepared and saved\n")

print(
    manifest_df[
        [
            "role",
            "sample",
            "duration_seconds",
            "transcript"
        ]
    ].to_string(index=False)
)

print(
    "\nSaved locally:",
    SELECTED_DIR
)

print(
    "Saved in Drive:",
    DRIVE_SELECTED_DIR
)

✅ Fixed samples prepared and saved

            role  sample  duration_seconds                                                                                                           transcript
          source       1          4.973515                                                     The monkey was planning to throw the net like the fishermen did.
          source       2          5.833515                                                 Gaze out to sea, picnic on benches and watch fishermen hook herring.
          source       3          6.893515 We all have our own methods of dealing with money, but here are a couple of saving tricks that will grow your money.
target_reference       0          6.474649                               Mercedes screamed, cried, laughed, and manifested the chaotic abandonment of hysteria.

Saved locally: /content/selected_hybrid_data
Saved in Drive: /content/drive/MyDrive/IJEEE_VC_data/selected_hybrid_data


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

OPENVOICE_REPO = Path(
    "/content/OpenVoice_repo"
)

OPENVOICE_MODEL_DIR = Path(
    "/content/drive/MyDrive/"
    "IJEEE_VC_data/models/OpenVoiceV2"
)


# -----------------------------------
# Clone OpenVoice source code
# -----------------------------------
if not (
    OPENVOICE_REPO /
    "openvoice/api.py"
).exists():

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/myshell-ai/OpenVoice.git",
            str(OPENVOICE_REPO)
        ],
        check=True
    )


# -----------------------------------
# Install without replacing Torch
# -----------------------------------
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        str(OPENVOICE_REPO),
        "--no-deps"
    ],
    check=True
)


# Install only required dependencies
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "librosa>=0.10.2",
        "soundfile",
        "unidecode",
        "eng_to_ipa",
        "inflect",
        "pypinyin",
        "cn2an",
        "jieba",
        "langid",
        "wavmark",
        "huggingface_hub"
    ],
    check=True
)


# -----------------------------------
# Download V2 converter to Drive
# -----------------------------------
from huggingface_hub import snapshot_download

OPENVOICE_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

snapshot_download(
    repo_id="myshell-ai/OpenVoiceV2",
    local_dir=str(OPENVOICE_MODEL_DIR),
    allow_patterns=["converter/*"]
)


# -----------------------------------
# Verify installation
# -----------------------------------
if str(OPENVOICE_REPO) not in sys.path:
    sys.path.insert(
        0,
        str(OPENVOICE_REPO)
    )

config_path = (
    OPENVOICE_MODEL_DIR /
    "converter/config.json"
)

checkpoint_path = (
    OPENVOICE_MODEL_DIR /
    "converter/checkpoint.pth"
)

print("\nOpenVoice source exists:", (
    OPENVOICE_REPO /
    "openvoice/api.py"
).exists())

print(
    "Config exists:",
    config_path.exists()
)

print(
    "Checkpoint exists:",
    checkpoint_path.exists()
)

from openvoice.api import ToneColorConverter

print("✅ OpenVoice import successful")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]


OpenVoice source exists: True
Config exists: True
Checkpoint exists: True


/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")


✅ OpenVoice import successful


In [ ]:
import time
import torch
from pathlib import Path

from openvoice.api import ToneColorConverter

OPENVOICE_MODEL_DIR = Path(
    "/content/drive/MyDrive/"
    "IJEEE_VC_data/models/OpenVoiceV2"
)

EMBEDDING_DIR = Path(
    "/content/drive/MyDrive/"
    "IJEEE_VC_data/openvoice_embeddings"
)

EMBEDDING_DIR.mkdir(
    parents=True,
    exist_ok=True
)

config_path = (
    OPENVOICE_MODEL_DIR /
    "converter/config.json"
)

checkpoint_path = (
    OPENVOICE_MODEL_DIR /
    "converter/checkpoint.pth"
)

device = (
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

print("Loading OpenVoice V2...")

tone_color_converter = ToneColorConverter(
    str(config_path),
    device=device
)

tone_color_converter.load_ckpt(
    str(checkpoint_path)
)

print("\n✅ OpenVoice V2 loaded")
print("Device:", device)
print("Version:", tone_color_converter.version)


def synchronize_gpu():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


SOURCE_PATHS = {
    1: "/content/selected_hybrid_data/joe_source_01.wav",
    2: "/content/selected_hybrid_data/joe_source_02.wav",
    3: "/content/selected_hybrid_data/joe_source_03.wav"
}

TARGET_PATH = (
    "/content/selected_hybrid_data/"
    "kathleen_reference.wav"
)


# -----------------------------------
# Kathleen target embedding
# -----------------------------------
print("\nExtracting Kathleen embedding...")

synchronize_gpu()
target_start = time.perf_counter()

target_se = tone_color_converter.extract_se(
    [TARGET_PATH],
    se_save_path=str(
        EMBEDDING_DIR /
        "kathleen_target_se.pth"
    )
)

synchronize_gpu()
target_embedding_time = (
    time.perf_counter() - target_start
)

print(
    "Kathleen embedding time:",
    round(target_embedding_time, 4),
    "seconds"
)


# -----------------------------------
# Joe source embeddings
# -----------------------------------
source_embeddings = {}
source_embedding_times = {}

print("\nExtracting Joe embeddings...")

for sample, source_path in SOURCE_PATHS.items():

    synchronize_gpu()
    start = time.perf_counter()

    source_embeddings[sample] = (
        tone_color_converter.extract_se(
            [source_path],
            se_save_path=str(
                EMBEDDING_DIR /
                f"joe_source_{sample:02d}_se.pth"
            )
        )
    )

    synchronize_gpu()
    elapsed = time.perf_counter() - start

    source_embedding_times[sample] = elapsed

    print(
        f"Sample {sample}:",
        round(elapsed, 4),
        "seconds"
    )


print("\n✅ Embeddings extracted and saved")
print("Saved in:", EMBEDDING_DIR)


Loading OpenVoice V2...


/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


step59000_snr39.99_pesq4.35_BERP_none0.3(…): reconstructing file:   0%|          |  0.00B / 10.0MB            

step59000_snr39.99_pesq4.35_BERP_none0.3(…): downloading bytes:           |  0.00B            

Loaded checkpoint '/content/drive/MyDrive/IJEEE_VC_data/models/OpenVoiceV2/converter/checkpoint.pth'
missing/unexpected keys: [] []

✅ OpenVoice V2 loaded
Device: cuda:0
Version: v2

Extracting Kathleen embedding...


/usr/local/lib/python3.13/dist-packages/torch/functional.py:681: UserWarning: stft with return_complex=False is deprecated. In a future pytorch release, stft will return complex tensors for all inputs, and return_complex=False will raise an error.
Note: you can still call torch.view_as_real on the complex output to recover the old return format. (Triggered internally at /pytorch/aten/src/ATen/native/SpectralOps.cpp:874.)
  return _VF.stft(  # type: ignore[attr-defined]


Kathleen embedding time: 14.8795 seconds

Extracting Joe embeddings...
Sample 1: 0.0394 seconds
Sample 2: 0.0195 seconds
Sample 3: 0.016 seconds

✅ Embeddings extracted and saved
Saved in: /content/drive/MyDrive/IJEEE_VC_data/openvoice_embeddings


In [ ]:
import time
import random
import shutil
import numpy as np
import pandas as pd
import soundfile as sf
import torch

from pathlib import Path


TAU_VALUES = [0.1, 0.3, 0.5, 0.7, 0.9]

LOCAL_OUTPUT_DIR = Path(
    "/content/openvoice_tau_experiment"
)

DRIVE_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/"
    "IJEEE_VC_data/openvoice_tau_experiment"
)

LOCAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DRIVE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def synchronize_gpu():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def set_fixed_seed(sample_number):
    # Same seed for each sample across all tau values
    seed = 4200 + sample_number

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# -----------------------------------
# Warm-up: excluded from measurements
# -----------------------------------
print("Running OpenVoice warm-up...")

warmup_output = (
    LOCAL_OUTPUT_DIR /
    "_openvoice_warmup.wav"
)

set_fixed_seed(1)

tone_color_converter.convert(
    audio_src_path=SOURCE_PATHS[1],
    src_se=source_embeddings[1],
    tgt_se=target_se,
    output_path=str(warmup_output),
    tau=0.3,
    message="@MyShell"
)

synchronize_gpu()

print("✅ Warm-up completed")


# -----------------------------------
# Tau sweep
# -----------------------------------
openvoice_results = []

for tau in TAU_VALUES:

    tau_dir = (
        LOCAL_OUTPUT_DIR /
        f"tau_{tau}"
    )

    tau_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    print("\n" + "=" * 60)
    print("Running tau =", tau)
    print("=" * 60)

    for sample_number in [1, 2, 3]:

        source_path = SOURCE_PATHS[
            sample_number
        ]

        output_path = (
            tau_dir /
            (
                f"openvoice_sample_"
                f"{sample_number:02d}_"
                f"tau_{tau}.wav"
            )
        )

        duration = sf.info(
            source_path
        ).duration

        set_fixed_seed(sample_number)

        synchronize_gpu()
        start = time.perf_counter()

        tone_color_converter.convert(
            audio_src_path=source_path,
            src_se=source_embeddings[
                sample_number
            ],
            tgt_se=target_se,
            output_path=str(output_path),
            tau=tau,
            message="@MyShell"
        )

        synchronize_gpu()
        conversion_time = (
            time.perf_counter() - start
        )

        embedding_time = (
            source_embedding_times[
                sample_number
            ]
        )

        total_latency = (
            embedding_time +
            conversion_time
        )

        rtf = total_latency / duration

        openvoice_results.append({
            "tau": tau,
            "sample": sample_number,
            "random_seed": (
                4200 + sample_number
            ),
            "source_file": source_path,
            "target_reference": TARGET_PATH,
            "output_file": str(output_path),
            "duration_seconds": duration,
            "source_embedding_seconds": (
                embedding_time
            ),
            "conversion_seconds": (
                conversion_time
            ),
            "openvoice_latency_seconds": (
                total_latency
            ),
            "openvoice_rtf": rtf
        })

        print(
            f"Sample {sample_number}: "
            f"latency={total_latency:.4f}s, "
            f"RTF={rtf:.4f}"
        )


# -----------------------------------
# Save detailed metrics
# -----------------------------------
openvoice_results_df = pd.DataFrame(
    openvoice_results
)

metrics_path = (
    LOCAL_OUTPUT_DIR /
    "openvoice_tau_metrics.csv"
)

openvoice_results_df.to_csv(
    metrics_path,
    index=False
)


# -----------------------------------
# Summary
# -----------------------------------
openvoice_summary = (
    openvoice_results_df
    .groupby("tau")
    .agg(
        mean_latency=(
            "openvoice_latency_seconds",
            "mean"
        ),
        std_latency=(
            "openvoice_latency_seconds",
            "std"
        ),
        mean_rtf=(
            "openvoice_rtf",
            "mean"
        ),
        std_rtf=(
            "openvoice_rtf",
            "std"
        )
    )
    .reset_index()
)

summary_path = (
    LOCAL_OUTPUT_DIR /
    "openvoice_tau_summary.csv"
)

openvoice_summary.to_csv(
    summary_path,
    index=False
)


# -----------------------------------
# Save outputs and metrics to Drive
# -----------------------------------
shutil.copytree(
    LOCAL_OUTPUT_DIR,
    DRIVE_OUTPUT_DIR,
    dirs_exist_ok=True
)

print("\n" + "=" * 60)
print("OPENVOICE TAU SUMMARY")
print("=" * 60)

print(
    openvoice_summary.round(4)
)

print(
    "\n✅ Outputs saved locally:",
    LOCAL_OUTPUT_DIR
)

print(
    "✅ Outputs saved in Drive:",
    DRIVE_OUTPUT_DIR
)

Running OpenVoice warm-up...
✅ Warm-up completed

Running tau = 0.1
Sample 1: latency=0.3563s, RTF=0.0716
Sample 2: latency=0.3418s, RTF=0.0586
Sample 3: latency=0.6405s, RTF=0.0929

Running tau = 0.3
Sample 1: latency=0.2877s, RTF=0.0579
Sample 2: latency=0.2873s, RTF=0.0492
Sample 3: latency=0.3167s, RTF=0.0459

Running tau = 0.5
Sample 1: latency=0.2935s, RTF=0.0590
Sample 2: latency=0.2824s, RTF=0.0484
Sample 3: latency=0.3134s, RTF=0.0455

Running tau = 0.7
Sample 1: latency=0.2871s, RTF=0.0577
Sample 2: latency=0.2840s, RTF=0.0487
Sample 3: latency=0.3110s, RTF=0.0451

Running tau = 0.9
Sample 1: latency=0.2885s, RTF=0.0580
Sample 2: latency=0.2849s, RTF=0.0488
Sample 3: latency=0.3145s, RTF=0.0456

OPENVOICE TAU SUMMARY
   tau  mean_latency  std_latency  mean_rtf  std_rtf
0  0.1        0.4462       0.1684    0.0744   0.0173
1  0.3        0.2972       0.0168    0.0510   0.0061
2  0.5        0.2964       0.0157    0.0510   0.0071
3  0.7        0.2940       0.0148    0.0505   0.006

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

SEED_DIR = Path("/content/seed-vc")

SEED_COMMIT = (
    "51383efd921027683c89e5348211d93ff12ac2a8"
)


# -----------------------------------
# Clone Seed-VC
# -----------------------------------
if not (
    SEED_DIR /
    ".git"
).exists():

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/Plachtaa/seed-vc.git",
            str(SEED_DIR)
        ],
        check=True
    )


# Use the exact paper commit
subprocess.run(
    [
        "git",
        "-C",
        str(SEED_DIR),
        "checkout",
        SEED_COMMIT
    ],
    check=True
)


# -----------------------------------
# Install inference dependencies
# Keep current Torch, NumPy and Librosa
# -----------------------------------
required_packages = [
    "accelerate",
    "munch==4.0.0",
    "einops==0.8.0",
    "hydra-core==1.3.2",
    "pyyaml",
    "python-dotenv",
    "transformers==4.46.3",
    "descript-audio-codec==1.0.0",
    "jiwer==3.0.3",
    "soundfile"
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q"
    ] + required_packages,
    check=True
)

print("\n✅ Seed-VC dependencies installed")
print("Repository:", SEED_DIR)
print("Commit:", SEED_COMMIT)

print("\nPython:", sys.version.split()[0])

print(
    "⚠️ الآن اعملي Runtime → Restart session"
)


✅ Seed-VC dependencies installed
Repository: /content/seed-vc
Commit: 51383efd921027683c89e5348211d93ff12ac2a8

Python: 3.13.15
⚠️ الآن اعملي Runtime → Restart session


In [ ]:
# Must be set before importing Transformers or Seed-VC
import os

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

from google.colab import drive
drive.mount("/content/drive")

import sys
import shutil
import torch
from pathlib import Path


PROJECT_DIR = Path(
    "/content/drive/MyDrive/IJEEE_VC_data"
)

SEED_DIR = Path("/content/seed-vc")

LOCAL_SELECTED_DIR = Path(
    "/content/selected_hybrid_data"
)

LOCAL_OPENVOICE_DIR = Path(
    "/content/openvoice_tau_experiment"
)


# -----------------------------------
# Restore fixed inputs from Drive
# -----------------------------------
shutil.copytree(
    PROJECT_DIR / "selected_hybrid_data",
    LOCAL_SELECTED_DIR,
    dirs_exist_ok=True
)

shutil.copytree(
    PROJECT_DIR / "openvoice_tau_experiment",
    LOCAL_OPENVOICE_DIR,
    dirs_exist_ok=True
)


# -----------------------------------
# Verify environment
# -----------------------------------
assert SEED_DIR.exists(), (
    "❌ Seed-VC repository is missing"
)

if str(SEED_DIR) not in sys.path:
    sys.path.insert(0, str(SEED_DIR))

os.chdir(SEED_DIR)

from transformers.utils import is_tf_available

openvoice_outputs = list(
    LOCAL_OPENVOICE_DIR.rglob(
        "openvoice_sample_*.wav"
    )
)

print("=" * 60)
print("POST-RESTART CHECK")
print("=" * 60)

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print(
    "TensorFlow enabled in Transformers:",
    is_tf_available()
)

print(
    "Selected inputs:",
    len(
        list(
            LOCAL_SELECTED_DIR.glob("*.wav")
        )
    )
)

print(
    "OpenVoice tau outputs:",
    len(openvoice_outputs)
)

print(
    "Seed repository exists:",
    SEED_DIR.exists()
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

POST-RESTART CHECK
Python: 3.13.15
Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
TensorFlow enabled in Transformers: False
Selected inputs: 4
OpenVoice tau outputs: 15
Seed repository exists: True


In [ ]:
import os
import sys
import torch

from pathlib import Path
from types import SimpleNamespace

SEED_DIR = Path("/content/seed-vc")

os.chdir(SEED_DIR)

if str(SEED_DIR) not in sys.path:
    sys.path.insert(0, str(SEED_DIR))

import inference as seed_inference


seed_baseline_args = SimpleNamespace(
    source=(
        "/content/openvoice_tau_experiment/"
        "tau_0.3/openvoice_sample_01_tau_0.3.wav"
    ),
    target=(
        "/content/selected_hybrid_data/"
        "kathleen_reference.wav"
    ),
    output="/content/seed_baseline_test",
    diffusion_steps=20,
    length_adjust=1.0,
    inference_cfg_rate=0.7,
    f0_condition=False,
    auto_f0_adjust=False,
    semi_tone_shift=0,
    checkpoint=None,
    config=None,
    fp16=True
)


print("Loading Seed-VC models...")
print(
    "The first load may download Seed, "
    "Whisper, CAM++ and BigVGAN.\n"
)

seed_model_bundle = (
    seed_inference.load_models(
        seed_baseline_args
    )
)


# Prevent model reloading during every conversion
seed_inference.load_models = (
    lambda args: seed_model_bundle
)


print("\n✅ Seed-VC models loaded and cached")
print(
    "Loaded components:",
    len(seed_model_bundle)
)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

Loading Seed-VC models...
The first load may download Seed, Whisper, CAM++ and BigVGAN.



DiT_seed_v2_uvit_whisper_small_wavenet_b(…):   0%|          | 0.00/440M [00:00<?, ?B/s]

(…)_mel_seed_uvit_whisper_small_wavenet.yml: 0.00B [00:00, ?B/s]

cfm loaded
length_regulator loaded


campplus_cn_common.bin:   0%|          | 0.00/28.0M [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

... (3 outputs left out)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]


✅ Seed-VC models loaded and cached
Loaded components: 7
GPU: Tesla T4


In [ ]:
import time
import random
import numpy as np
import torch

from pathlib import Path
from types import SimpleNamespace
from IPython.display import Audio, display


def synchronize_gpu():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def set_seed_for_sample(sample_number):
    seed = 5200 + sample_number

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


TEST_SOURCE = (
    "/content/openvoice_tau_experiment/"
    "tau_0.3/openvoice_sample_01_tau_0.3.wav"
)

TARGET_REFERENCE = (
    "/content/selected_hybrid_data/"
    "kathleen_reference.wav"
)


# -----------------------------------
# Warm-up: excluded from measurements
# -----------------------------------
warmup_args = SimpleNamespace(
    source=TEST_SOURCE,
    target=TARGET_REFERENCE,
    output="/content/seed_warmup",
    diffusion_steps=20,
    length_adjust=1.0,
    inference_cfg_rate=0.7,
    f0_condition=False,
    auto_f0_adjust=False,
    semi_tone_shift=0,
    checkpoint=None,
    config=None,
    fp16=True
)

print("Running Seed-VC warm-up...")

set_seed_for_sample(1)

seed_inference.main(
    warmup_args
)

synchronize_gpu()

print("✅ Warm-up completed")


# -----------------------------------
# Measured baseline test
# -----------------------------------
test_output_dir = Path(
    "/content/hybrid_baseline_test"
)

test_output_dir.mkdir(
    parents=True,
    exist_ok=True
)

test_args = SimpleNamespace(
    source=TEST_SOURCE,
    target=TARGET_REFERENCE,
    output=str(test_output_dir),
    diffusion_steps=20,
    length_adjust=1.0,
    inference_cfg_rate=0.7,
    f0_condition=False,
    auto_f0_adjust=False,
    semi_tone_shift=0,
    checkpoint=None,
    config=None,
    fp16=True
)

set_seed_for_sample(1)

synchronize_gpu()
start = time.perf_counter()

seed_inference.main(
    test_args
)

synchronize_gpu()
seed_latency = time.perf_counter() - start

generated_files = sorted(
    test_output_dir.glob("*.wav")
)

assert generated_files, (
    "❌ Seed-VC did not produce an output"
)

test_output = generated_files[-1]

print("\n" + "=" * 60)
print("HYBRID BASELINE TEST")
print("=" * 60)

print("OpenVoice tau:", 0.3)
print("Seed diffusion steps:", 20)
print(
    "Measured Seed latency:",
    round(seed_latency, 4),
    "seconds"
)
print("Final output:", test_output)

print("\nFinal hybrid audio:")
display(
    Audio(str(test_output))
)

Running Seed-VC warm-up...


It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
100%|██████████| 20/20 [00:01<00:00, 14.80it/s]


RTF: 0.8477697357813889


It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


✅ Warm-up completed


100%|██████████| 20/20 [00:00<00:00, 20.51it/s]


RTF: 0.3291084179785277

HYBRID BASELINE TEST
OpenVoice tau: 0.3
Seed diffusion steps: 20
Measured Seed latency: 1.64 seconds
Final output: /content/hybrid_baseline_test/vc_openvoice_sample_01_tau_0_kathleen_reference_1.0_20_0.7.wav

Final hybrid audio:


In [ ]:
import time
import random
import shutil
import numpy as np
import pandas as pd
import soundfile as sf
import torch

from pathlib import Path
from types import SimpleNamespace


TAU_VALUES = [0.1, 0.3, 0.5, 0.7, 0.9]
SAMPLE_NUMBERS = [1, 2, 3]

TARGET_REFERENCE = (
    "/content/selected_hybrid_data/"
    "kathleen_reference.wav"
)

OPENVOICE_METRICS_PATH = (
    "/content/openvoice_tau_experiment/"
    "openvoice_tau_metrics.csv"
)

LOCAL_HYBRID_DIR = Path(
    "/content/hybrid_tau_experiment"
)

DRIVE_HYBRID_DIR = Path(
    "/content/drive/MyDrive/"
    "IJEEE_VC_data/hybrid_tau_experiment"
)

LOCAL_HYBRID_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DRIVE_HYBRID_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def synchronize_gpu():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def set_seed_for_sample(sample_number):
    # Same Seed-VC randomness for the same sample
    # across all OpenVoice tau values
    seed = 5200 + sample_number

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# -----------------------------------
# Load OpenVoice timing measurements
# -----------------------------------
openvoice_metrics = pd.read_csv(
    OPENVOICE_METRICS_PATH
)

openvoice_lookup = {}

for _, row in openvoice_metrics.iterrows():

    key = (
        round(float(row["tau"]), 1),
        int(row["sample"])
    )

    openvoice_lookup[key] = {
        "latency": float(
            row["openvoice_latency_seconds"]
        ),
        "rtf": float(
            row["openvoice_rtf"]
        ),
        "duration": float(
            row["duration_seconds"]
        )
    }


# -----------------------------------
# Full hybrid experiment
# -----------------------------------
hybrid_results = []

for tau in TAU_VALUES:

    print("\n" + "=" * 70)
    print(
        f"Running hybrid outputs for tau={tau}"
    )
    print("=" * 70)

    for sample_number in SAMPLE_NUMBERS:

        openvoice_source = (
            "/content/openvoice_tau_experiment/"
            f"tau_{tau}/"
            f"openvoice_sample_"
            f"{sample_number:02d}_tau_{tau}.wav"
        )

        assert Path(openvoice_source).exists(), (
            f"❌ Missing: {openvoice_source}"
        )

        local_output_dir = (
            LOCAL_HYBRID_DIR /
            f"tau_{tau}" /
            f"sample_{sample_number:02d}"
        )

        drive_output_dir = (
            DRIVE_HYBRID_DIR /
            f"tau_{tau}" /
            f"sample_{sample_number:02d}"
        )

        local_output_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        drive_output_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        args = SimpleNamespace(
            source=openvoice_source,
            target=TARGET_REFERENCE,
            output=str(local_output_dir),
            diffusion_steps=20,
            length_adjust=1.0,
            inference_cfg_rate=0.7,
            f0_condition=False,
            auto_f0_adjust=False,
            semi_tone_shift=0,
            checkpoint=None,
            config=None,
            fp16=True
        )

        set_seed_for_sample(
            sample_number
        )

        synchronize_gpu()
        seed_start = time.perf_counter()

        seed_inference.main(args)

        synchronize_gpu()
        seed_latency = (
            time.perf_counter() -
            seed_start
        )

        generated_files = sorted(
            local_output_dir.glob("*.wav")
        )

        assert generated_files, (
            f"❌ No output for tau={tau}, "
            f"sample={sample_number}"
        )

        final_output = generated_files[-1]

        # Save this output immediately to Drive
        drive_output = (
            drive_output_dir /
            final_output.name
        )

        shutil.copy2(
            final_output,
            drive_output
        )

        timing = openvoice_lookup[
            (
                round(tau, 1),
                sample_number
            )
        ]

        original_duration = timing[
            "duration"
        ]

        openvoice_latency = timing[
            "latency"
        ]

        hybrid_latency = (
            openvoice_latency +
            seed_latency
        )

        seed_rtf = (
            seed_latency /
            original_duration
        )

        hybrid_rtf = (
            hybrid_latency /
            original_duration
        )

        final_duration = sf.info(
            str(final_output)
        ).duration

        hybrid_results.append({
            "tau": tau,
            "sample": sample_number,
            "openvoice_seed": (
                4200 + sample_number
            ),
            "seed_vc_seed": (
                5200 + sample_number
            ),
            "diffusion_steps": 20,
            "length_adjust": 1.0,
            "inference_cfg_rate": 0.7,
            "f0_condition": False,
            "fp16": True,
            "original_duration_seconds": (
                original_duration
            ),
            "openvoice_latency_seconds": (
                openvoice_latency
            ),
            "seed_latency_seconds": (
                seed_latency
            ),
            "hybrid_latency_seconds": (
                hybrid_latency
            ),
            "seed_rtf": seed_rtf,
            "hybrid_rtf": hybrid_rtf,
            "final_duration_seconds": (
                final_duration
            ),
            "openvoice_output": (
                openvoice_source
            ),
            "local_hybrid_output": (
                str(final_output)
            ),
            "drive_hybrid_output": (
                str(drive_output)
            )
        })

        # Checkpoint metrics after every output
        checkpoint_df = pd.DataFrame(
            hybrid_results
        )

        checkpoint_df.to_csv(
            LOCAL_HYBRID_DIR /
            "hybrid_tau_metrics.csv",
            index=False
        )

        checkpoint_df.to_csv(
            DRIVE_HYBRID_DIR /
            "hybrid_tau_metrics.csv",
            index=False
        )

        print(
            f"Sample {sample_number}: "
            f"Seed={seed_latency:.4f}s, "
            f"Hybrid={hybrid_latency:.4f}s, "
            f"RTF={hybrid_rtf:.4f}"
        )


# -----------------------------------
# Final summary
# -----------------------------------
hybrid_results_df = pd.DataFrame(
    hybrid_results
)

hybrid_summary = (
    hybrid_results_df
    .groupby("tau")
    .agg(
        mean_seed_latency=(
            "seed_latency_seconds",
            "mean"
        ),
        std_seed_latency=(
            "seed_latency_seconds",
            "std"
        ),
        mean_hybrid_latency=(
            "hybrid_latency_seconds",
            "mean"
        ),
        std_hybrid_latency=(
            "hybrid_latency_seconds",
            "std"
        ),
        mean_hybrid_rtf=(
            "hybrid_rtf",
            "mean"
        ),
        std_hybrid_rtf=(
            "hybrid_rtf",
            "std"
        )
    )
    .reset_index()
)

hybrid_summary.to_csv(
    LOCAL_HYBRID_DIR /
    "hybrid_tau_summary.csv",
    index=False
)

hybrid_summary.to_csv(
    DRIVE_HYBRID_DIR /
    "hybrid_tau_summary.csv",
    index=False
)

print("\n" + "=" * 75)
print("FINAL HYBRID TAU SUMMARY")
print("=" * 75)

print(
    hybrid_summary.round(4)
)

print(
    "\n✅ All outputs and metrics saved to:",
    DRIVE_HYBRID_DIR
)

It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.



Running hybrid outputs for tau=0.1


100%|██████████| 20/20 [00:01<00:00, 15.50it/s]
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


RTF: 0.4462856774340236
Sample 1: Seed=2.2308s, Hybrid=2.5870s, RTF=0.5202


100%|██████████| 20/20 [00:01<00:00, 16.02it/s]
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


RTF: 0.374159284782006
Sample 2: Seed=2.1856s, Hybrid=2.5274s, RTF=0.4332


... (22 outputs left out)

100%|██████████| 20/20 [00:01<00:00, 17.36it/s]
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


RTF: 0.3519954548525323
Sample 2: Seed=2.0571s, Hybrid=2.3420s, RTF=0.4015


100%|██████████| 20/20 [00:01<00:00, 16.91it/s]


RTF: 0.3235210822408306
Sample 3: Seed=2.2349s, Hybrid=2.5494s, RTF=0.3698

FINAL HYBRID TAU SUMMARY
   tau  mean_seed_latency  std_seed_latency  mean_hybrid_latency  \
0  0.1             2.4015            0.3355               2.8476   
1  0.3             2.7384            0.9127               3.0356   
2  0.5             2.4126            0.1659               2.7090   
3  0.7             2.0674            0.2095               2.3614   
4  0.9             2.0360            0.2102               2.3320   

   std_hybrid_latency  mean_hybrid_rtf  std_hybrid_rtf  
0              0.5040           0.4836          0.0451  
1              0.9282           0.5066          0.0728  
2              0.1655           0.4698          0.1020  
3              0.2201           0.4033          0.0284  
4              0.2226           0.3981          0.0268  

✅ All outputs and metrics saved to: /content/drive/MyDrive/IJEEE_VC_data/hybrid_tau_experiment


In [ ]:
!pip install -q speechbrain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.1/782.1 kB 53.1 MB/s eta 0:00:00


In [ ]:
BEST_TAU = 0.5
CFG_RATE = 0.7
LENGTH_ADJUST = 1.0

In [ ]:
import os
import sys
import glob
import torch

from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"

OPENVOICE_DIR = os.path.join(
    BASE_DIR,
    "openvoice_tau_experiment"
)

TARGET_PATH = os.path.join(
    BASE_DIR,
    "selected_hybrid_data",
    "kathleen_reference.wav"
)

SEED_REPO = "/content/seed-vc"

# البحث عن مخرجات OpenVoice الخاصة بـ tau=0.5
tau_05_files = sorted(
    glob.glob(
        os.path.join(
            OPENVOICE_DIR,
            "**",
            "*tau_0.5*.wav"
        ),
        recursive=True
    )
)

print("=" * 65)
print("SEED STEPS EXPERIMENT — PRE-CHECK")
print("=" * 65)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Seed repository exists:", os.path.isdir(SEED_REPO))
print("Kathleen reference exists:", os.path.isfile(TARGET_PATH))
print("Seed models still loaded:", "seed_model_bundle" in globals())

print("\nOpenVoice tau=0.5 inputs:", len(tau_05_files))

for path in tau_05_files:
    print(" -", path)

assert torch.cuda.is_available(), "❌ لازم تشغلي الجلسة على GPU"
assert os.path.isdir(SEED_REPO), "❌ Seed-VC repository is missing"
assert os.path.isfile(TARGET_PATH), "❌ Kathleen reference is missing"
assert len(tau_05_files) == 3, (
    f"❌ Expected exactly 3 tau=0.5 files, found {len(tau_05_files)}"
)

print("\n✅ كل ملفات تجربة Seed موجودة")

Mounted at /content/drive
SEED STEPS EXPERIMENT — PRE-CHECK
CUDA available: True
GPU: Tesla T4
Seed repository exists: False
Kathleen reference exists: True
Seed models still loaded: False

OpenVoice tau=0.5 inputs: 3
 - /content/drive/MyDrive/IJEEE_VC_data/openvoice_tau_experiment/tau_0.5/openvoice_sample_01_tau_0.5.wav
 - /content/drive/MyDrive/IJEEE_VC_data/openvoice_tau_experiment/tau_0.5/openvoice_sample_02_tau_0.5.wav
 - /content/drive/MyDrive/IJEEE_VC_data/openvoice_tau_experiment/tau_0.5/openvoice_sample_03_tau_0.5.wav


AssertionError: ❌ Seed-VC repository is missing